In [ ]:
"""
HELF - Hybrid Ensemble Learning Framework for UPI Fraud Detection
=================================================================
A single-file implementation of:
  Abhilasha, Vijayakarthik & Pol, "A Hybrid Ensemble Learning Framework for
  Real-Time Fraud Detection and Risk Scoring in UPI Transactions",
  IJRESM 9(8):36-42, 2026.
The idea in one line:
  Run 4 models on every transaction, mix their outputs into ONE risk score,
  then turn that score into one of 3 verdicts.

  The paper mixes them with fixed weights (Equation 1):

      S = 0.70 * (0.20*LR + 0.45*RF + 0.35*XGB)  +  0.30 * IsolationForest

  With fixed weights the weak models (LR, Isolation Forest) pull the score
  down, so the mix ends up worse than the random forest alone. This version
  LEARNS the mix instead: a small meta-model is trained on the validation set
  and sees the 4 model scores plus the transaction features. That is called
  stacking. The final score is

      S = 0.90 * learned score  +  0.10 * Equation 1 score

  The plain Equation 1 score is also shown in the results table, so the two
  can be compared.

      S below the approve line  -> Legitimate  (approve)
      S in between              -> Suspicious  (send to a human)
      S above the block line    -> Fraudulent  (block)

How to run:
      pip install numpy pandas scikit-learn matplotlib
      python helf_simple.py

Everything is in this one file, in the order it actually happens:
  STEP 1  make the dataset
  STEP 2  clean it
  STEP 3  split it (train / validation / test)
  STEP 4  prepare features (impute, encode, scale, SMOTE)
  STEP 5  train the 4 models
  STEP 6  fuse their scores (learned fusion, plus the paper's fixed weights)
  STEP 7  pick the thresholds
  STEP 8  measure how well it worked
  STEP 9  draw the charts
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")                      # save charts to files, no pop-up window
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, confusion_matrix)

# XGBoost if you have it, otherwise scikit-learn's boosting (same settings)
try:
    from xgboost import XGBClassifier
    HAVE_XGBOOST = True
except ImportError:
    HAVE_XGBOOST = False

SEED = 42
rng = np.random.default_rng(SEED)

# ---- all the numbers from the paper, in one place ----------------------------
N_RECORDS = 150000        # raw transactions we generate
FRAUD_RATE = 0.086         # ~8% fraud, like real UPI traffic
ALPHA = 0.70               # how much we trust the supervised models
W_LR, W_RF, W_XGB = 0.20, 0.45, 0.35      # weights inside the supervised part
MAX_BLOCK_FPR = 0.04       # never auto-block more than 4% of good customers
META_ROUNDS, META_DEPTH = 200, 4          # size of the meta-model that learns the fusion
META_FOLDS = 5             # validation is cut into this many parts to train it
BETA = 0.90                # final score = BETA * learned score + (1 - BETA) * Equation 1


# ==============================================================================
# STEP 1 - MAKE THE DATASET
# ==============================================================================
# The paper used a private Kaggle file, so we build a stand-in with the same
# shape: 150,000 rows, ~8% fraud, and the same columns it describes.

def make_normal_transactions(n):

    return pd.DataFrame({
        "amount":                np.round(np.exp(rng.normal(6.1, 1.15, n)), 2),
        "txn_type":              rng.choice(["p2p", "merchant", "bill", "recharge"],n, p=[.46, .34, .12, .08]),

        "hour":                  np.clip(rng.normal(14, 4.5, n) % 24, 0, 23).astype(int),
        "day_of_week":           rng.integers(0, 7, n),

        "txn_freq_24h":          rng.poisson(3.2, n),
        "amount_to_avg_ratio":   np.round(np.exp(rng.normal(0, 0.45, n)), 3),
        "account_age_days":      rng.integers(90, 3000, n),
        "distinct_payees_7d":    rng.poisson(4.0, n),

        "recipient_reputation":  np.round(rng.beta(6, 2, n), 3),
        "second_degree_links":   rng.poisson(2.0, n),

        "device_match":          (rng.random(n) < 0.955).astype(int),
        "vpa_verified":          (rng.random(n) < 0.93).astype(int),
        "new_payee":             (rng.random(n) < 0.17).astype(int),
        "collect_request":       (rng.random(n) < 0.09).astype(int),
    })


def make_fraud_transactions(n, kind):
    normal = make_normal_transactions(n)
    fraud = make_normal_transactions(n)

    if kind == "big":
        strength = rng.beta(5.0, 2.5, n) * 0.65        # mostly obvious
        fraud["amount"] = np.round(np.exp(rng.normal(9.4, 0.9, n)), 2)
        fraud["hour"] = rng.choice(np.r_[0:5, 22:24], n)
        fraud["amount_to_avg_ratio"] = np.round(np.exp(rng.normal(2.6, 0.6, n)), 3)
        fraud["txn_freq_24h"] = rng.poisson(11.0, n)
        fraud["account_age_days"] = rng.integers(1, 400, n)
        fraud["distinct_payees_7d"] = rng.poisson(12.0, n)
        fraud["recipient_reputation"] = np.round(rng.beta(1.4, 7, n), 3)
        fraud["second_degree_links"] = rng.poisson(8.5, n)
        fraud["device_match"] = (rng.random(n) < 0.10).astype(int)
        fraud["vpa_verified"] = (rng.random(n) < 0.22).astype(int)
        fraud["new_payee"] = 1
    else:
        strength = rng.beta(1.8, 3.2, n) * 0.65
        fraud["amount"] = np.round(np.exp(rng.normal(7.4, 1.0, n)), 2)
        fraud["amount_to_avg_ratio"] = np.round(np.exp(rng.normal(1.05, 0.55, n)), 3)
        fraud["txn_freq_24h"] = rng.poisson(5.5, n)
        fraud["distinct_payees_7d"] = rng.poisson(7.0, n)
        fraud["recipient_reputation"] = np.round(rng.beta(2.2, 5.0, n), 3)
        fraud["second_degree_links"] = rng.poisson(5.0, n)
        fraud["device_match"] = (rng.random(n) < 0.68).astype(int)
        fraud["vpa_verified"] = (rng.random(n) < 0.44).astype(int)
        fraud["new_payee"] = 1
        fraud["collect_request"] = (rng.random(n) < 0.63).astype(int)


    pick_one = ["txn_type", "device_match", "vpa_verified", "new_payee", "collect_request"]
    out = normal.copy()
    for col in out.columns:
        if col in pick_one:
            out[col] = np.where(rng.random(n) < strength, fraud[col], normal[col])
        else:
            out[col] = normal[col] + strength * (fraud[col] - normal[col])
    return out


def build_dataset():
    n_fraud = int(N_RECORDS * FRAUD_RATE)
    n_big = int(n_fraud * 0.42)
    n_social = n_fraud - n_big

    df = pd.concat([
        make_normal_transactions(N_RECORDS - n_fraud),
        make_fraud_transactions(n_big, "big"),
        make_fraud_transactions(n_social, "social"),
    ], ignore_index=True)

    df["is_fraud"] = np.r_[np.zeros(N_RECORDS - n_fraud), np.ones(n_fraud)].astype(int)

    unreported = rng.choice(np.where(df.is_fraud == 1)[0], 600, replace=False)
    df.loc[unreported, "is_fraud"] = 0


    for col in ["amount_to_avg_ratio", "recipient_reputation"]:
        df.loc[rng.choice(len(df), 2500, replace=False), col] = np.nan

    broken = rng.choice(len(df), 2618, replace=False)
    df.loc[broken[:1300], "amount"] = -1
    df.loc[broken[1300:], "hour"] = 99

    return df.sample(frac=1, random_state=SEED).reset_index(drop=True)



df = build_dataset()

import os
csv_path = "/content/synthetic_fraud_dataset.csv" if os.path.isdir("/content") \
    else "synthetic_fraud_dataset.csv"
df.to_csv(csv_path, index=False)

print(f"Saved to: {csv_path}")
print("Shape:", df.shape)

try:                                   # download button only exists in Colab
    from google.colab import files
    files.download(csv_path)
except ImportError:
    pass


In [ ]:
# =========================================================================
# STEP 2 - CLEAN
# =========================================================================
def clean(df):
    """Throw away rows that could not have happened."""
    good = (df.amount > 0) & df.hour.between(0, 23)
    return df[good].reset_index(drop=True)


# =========================================================================
# STEP 4 (helper) - SMOTE
# =========================================================================
def smote(X, y):
    """Create extra fraud examples so training sees a balanced 50/50 mix.

    For each real fraud row, pick one of its 5 nearest fraud neighbours and
    invent a new point somewhere on the line between them.

    Important: this is only ever applied to TRAINING data. If you balance the
    test set too, your accuracy becomes meaningless.
    """
    fraud_rows = X[y == 1]
    n_needed = (y == 0).sum() - (y == 1).sum()

    neighbours = NearestNeighbors(n_neighbors=6).fit(fraud_rows)
    _, idx = neighbours.kneighbors(fraud_rows)

    start = rng.integers(0, len(fraud_rows), n_needed)
    end = idx[start, rng.integers(1, 6, n_needed)]
    step = rng.random((n_needed, 1))

    new_rows = fraud_rows[start] + step * (fraud_rows[end] - fraud_rows[start])
    return np.vstack([X, new_rows]), np.r_[y, np.ones(n_needed, dtype=int)]


# =========================================================================
# MAIN
# =========================================================================
def main():
    print("STEP 1-2  building and cleaning the dataset")
    df = clean(build_dataset())
    y = df.is_fraud.to_numpy()
    print(f"          {len(df):,} usable transactions, {y.mean():.1%} fraud")

    # ---------------------------------------------------------------------
    # STEP 3 - SPLIT
    # ---------------------------------------------------------------------
    # 70% train (learn), 15% validation (tune), 15% test (final score, used once).
    # `stratify` keeps the same fraud rate in all three parts.
    print("STEP 3    splitting 70 / 15 / 15")
    rest, test = train_test_split(np.arange(len(df)), test_size=0.15,
                                  stratify=y, random_state=SEED)
    train, val = train_test_split(rest, test_size=0.1765,
                                  stratify=y[rest], random_state=SEED)

    # ---------------------------------------------------------------------
    # STEP 4 - PREPARE FEATURES
    # ---------------------------------------------------------------------
    print("STEP 4    imputing, encoding, scaling, SMOTE")
    features = df.drop(columns=["is_fraud"])
    features = pd.get_dummies(features, columns=["txn_type"])
    X = features.to_numpy(dtype=float)

    X_train, X_val, X_test = X[train], X[val], X[test]
    y_train, y_val, y_test = y[train], y[val], y[test]

    median = np.nanmedian(X_train, axis=0)
    for part in (X_train, X_val, X_test):
        gaps = np.isnan(part)
        part[gaps] = np.take(median, np.where(gaps)[1])

    mean, std = X_train.mean(0), X_train.std(0)
    std[std == 0] = 1
    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    X_train_real = X_train.copy()
    X_train, y_train = smote(X_train, y_train)
    print(f"          training rows after SMOTE: {len(y_train):,} (50% fraud)")

    # ---------------------------------------------------------------------
    # STEP 5 - TRAIN THE FOUR MODELS
    # ---------------------------------------------------------------------
    print("STEP 5    training 4 models")

    print("          1/4 logistic regression")
    lr = LogisticRegression(max_iter=1000, random_state=SEED)
    lr.fit(X_train, y_train)

    print("          2/4 random forest (100 trees)")
    rf = RandomForestClassifier(n_estimators=100, max_samples=0.7, n_jobs=-1, random_state=SEED)
    rf.fit(X_train, y_train)

    if HAVE_XGBOOST:
        print("          3/4 XGBoost (100 rounds, depth 4)")
        xgb = XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.02,
                            eval_metric="logloss", random_state=SEED)
    else:
        print("          3/4 gradient boosting (300 rounds, depth 4)")
        xgb = HistGradientBoostingClassifier(max_iter=300, max_depth=4,
                                             learning_rate=0.05, random_state=SEED)
    xgb.fit(X_train, y_train)

    print("          4/4 isolation forest (no labels)")
    # This one never sees the answers - it just looks for weird rows.
    # It must be trained on the ORIGINAL data, not the SMOTE-balanced version:
    # if half your data is fraud, nothing looks unusual any more.
    iso = IsolationForest(n_estimators=150, contamination=0.08,
                          n_jobs=-1, random_state=SEED)
    iso.fit(X_train_real)
    iso_low = (-iso.score_samples(X_train_real)).min()
    iso_high = (-iso.score_samples(X_train_real)).max()

    # ---------------------------------------------------------------------
    # STEP 6 - FUSE THE FOUR SCORES INTO ONE
    # ---------------------------------------------------------------------
    def model_scores(data):
        """The four raw scores for every row, each between 0 and 1."""
        p_lr = lr.predict_proba(data)[:, 1]
        p_rf = rf.predict_proba(data)[:, 1]
        p_xgb = xgb.predict_proba(data)[:, 1]
        # squeeze the anomaly score into 0..1 so it can sit beside probabilities
        a_if = np.clip((-iso.score_samples(data) - iso_low) / (iso_high - iso_low), 0, 1)
        return {"LR": p_lr, "RF": p_rf, "XGB": p_xgb, "IF": a_if}

    def fixed_weight_score(p):
        """Equation 1 in the paper: the same hand-picked weights for every row."""
        supervised = W_LR * p["LR"] + W_RF * p["RF"] + W_XGB * p["XGB"]
        return ALPHA * supervised + (1 - ALPHA) * p["IF"]

    def meta_inputs(p, data):
        """What the meta-model sees: the 4 scores, then the features."""
        return np.column_stack([p["LR"], p["RF"], p["XGB"], p["IF"], data])

    print("STEP 6    learning the fusion on the validation set")
    val_parts = model_scores(X_val)
    parts = model_scores(X_test)
    val_parts["EQ1"] = fixed_weight_score(val_parts)
    parts["EQ1"] = fixed_weight_score(parts)

    # Why the validation set: the 4 models have already memorised the training
    # rows, so their scores there look better than they really are. Validation
    # rows are new to them, so the scores are honest.
    #
    # The thresholds in STEP 7 are also picked on validation. To keep that fair,
    # every validation row must be scored by a meta-model that did NOT train on
    # it. So: split validation into 5 folds, train on 4, score the 5th, repeat.
    # That gives 5 meta-models. On the test set we average all 5.
    Z_val = meta_inputs(val_parts, X_val)
    Z_test = meta_inputs(parts, X_test)
    learned_val = np.zeros(len(y_val))
    learned_test = np.zeros(len(y_test))
    folds = StratifiedKFold(n_splits=META_FOLDS, shuffle=True, random_state=SEED)
    for fit_rows, held_out in folds.split(Z_val, y_val):
        meta = HistGradientBoostingClassifier(max_iter=META_ROUNDS, max_depth=META_DEPTH,
                                              learning_rate=0.05, random_state=SEED)
        meta.fit(Z_val[fit_rows], y_val[fit_rows])
        learned_val[held_out] = meta.predict_proba(Z_val[held_out])[:, 1]
        learned_test += meta.predict_proba(Z_test)[:, 1] / META_FOLDS

    # Final HELF score: mostly the learned score, with a little of Equation 1
    # kept in. The meta-model is very sure of itself (lots of scores at exactly
    # 0 or 1), and the Equation 1 part spreads those out again so the
    # three-tier lines in STEP 7 have something to work with.
    val_parts["S"] = BETA * learned_val + (1 - BETA) * val_parts["EQ1"]
    parts["S"] = BETA * learned_test + (1 - BETA) * parts["EQ1"]
    s_val = val_parts["S"]

    # ---------------------------------------------------------------------
    # STEP 7 - CHOOSE THE THRESHOLDS (on validation, never on test)
    # ---------------------------------------------------------------------
    print("STEP 7    calibrating thresholds on the validation set")

    # Upper line: block as much as we can without blocking more than 4% of
    # good customers.
    fpr, tpr, cuts = roc_curve(y_val, s_val)
    block_line = float(cuts[np.where(fpr <= MAX_BLOCK_FPR)[0][-1]])

    # Lower line: put it low enough that almost no fraud (<=1%) falls below it.
    # Anything above it but below the block line goes to a human instead of
    # being quietly approved. THIS is what drives false negatives toward zero.
    fraud_scores = np.sort(s_val[y_val == 1])
    approve_line = float(min(fraud_scores[int(0.01 * len(fraud_scores))],
                             block_line * 0.9))

    # One more line, only for the fair comparison table: the plain yes/no cut.
    best_cut, best_f1 = 0.5, 0
    for cut in np.linspace(0.05, 0.95, 100):
        f1 = f1_score(y_val, s_val >= cut, zero_division=0)
        if f1 > best_f1:
            best_f1, best_cut = f1, cut

    print(f"          approve below {approve_line:.3f} | "
          f"block above {block_line:.3f} | yes-no cut {best_cut:.3f}")

    # ---------------------------------------------------------------------
    # STEP 8 - MEASURE (test set, touched for the first time here)
    # ---------------------------------------------------------------------
    print("STEP 8    scoring the test set\n")
    S = parts["S"]

    def report(name, score, cut):
        pred = (score >= cut).astype(int)
        return {
            "Model": name,
            "Acc %": round(100 * accuracy_score(y_test, pred), 1),
            "Prec %": round(100 * precision_score(y_test, pred, zero_division=0), 1),
            "Recall %": round(100 * recall_score(y_test, pred, zero_division=0), 1),
            "F1 %": round(100 * f1_score(y_test, pred, zero_division=0), 1),
            "AUC": round(roc_auc_score(y_test, score), 3),
            "Missed frauds": int(((pred == 0) & (y_test == 1)).sum()),
        }

    # Each model gets its own best cut from validation, so the comparison is fair.

    def best_cut_for(key):
        cuts = np.linspace(0.05, 0.95, 100)
        return cuts[np.argmax([f1_score(y_val, val_parts[key] >= c, zero_division=0)
                               for c in cuts])]

    vote_test = (parts["LR"] >= .5).astype(int) + (parts["RF"] >= .5).astype(int) \
        + (parts["XGB"] >= .5).astype(int)

    table = pd.DataFrame([
        report("Logistic Regression", parts["LR"], best_cut_for("LR")),
        report("Random Forest", parts["RF"], best_cut_for("RF")),
        report("XGBoost", parts["XGB"], best_cut_for("XGB")),
        report("Isolation Forest", parts["IF"], best_cut_for("IF")),
        report("Majority Vote", vote_test / 3, 0.5),
        report("Fixed weights (Eq. 1)", parts["EQ1"], best_cut_for("EQ1")),
        report("HELF (learned fusion)", S, best_cut),
    ])
    print("TABLE 1 - test set, {:,} transactions, {:,} of them fraud"
          .format(len(y_test), int(y_test.sum())))
    print(table.to_string(index=False))

    # detection rate when we allow only 3% false alarms
    f, t, _ = roc_curve(y_test, S)
    print(f"\nAt a 3% false-alarm budget, HELF catches "
          f"{100 * np.interp(0.03, f, t):.1f}% of fraud")

    # --- the three-tier verdicts ---
    verdict = np.where(S < approve_line, "Legitimate",
                       np.where(S > block_line, "Fraudulent", "Suspicious"))
    print("\nTHREE-TIER VERDICTS")
    for name in ["Legitimate", "Suspicious", "Fraudulent"]:
        picked = verdict == name
        print(f"  {name:<12} {picked.sum():>6,} transactions "
              f"({100 * picked.mean():4.1f}%) - {int(y_test[picked].sum()):>5,} of them fraud")

    slipped = int(((verdict == "Legitimate") & (y_test == 1)).sum())
    print(f"\n  Fraud that slipped through as Legitimate: {slipped} out of {int(y_test.sum()):,}"
          f"  ({100 * (1 - slipped / y_test.sum()):.2f}% caught)")
    blocked_good = int(((verdict == "Fraudulent") & (y_test == 0)).sum())
    print(f"  Honest customers blocked by mistake:     {blocked_good:,}"
          f"  <- the real price of catching everything")

    # --- what mattered most ---
    names = features.columns.to_numpy()
    order = np.argsort(rf.feature_importances_)[::-1][:6]
    print("\nMOST USEFUL FEATURES (from the random forest)")
    for i in order:
        print(f"  {names[i]:<22} {rf.feature_importances_[i]:.3f}")

    # ---------------------------------------------------------------------
    # STEP 9 - CHARTS
    # ---------------------------------------------------------------------
    print("\nSTEP 9    saving charts")

    plt.figure(figsize=(3, 2))
    for label, score in [("Logistic Regression", parts["LR"]),
                         ("Random Forest", parts["RF"]),
                         ("XGBoost", parts["XGB"]),
                         ("HELF", S)]:
        f, t, _ = roc_curve(y_test, score)
        plt.plot(f, t, lw=2 if label == "HELF" else 1,
                 label=f"{label} (AUC {roc_auc_score(y_test, score):.3f})")
    plt.plot([0, 1], [0, 1], "k--", lw=0.8)
    plt.xlabel("false alarms"); plt.ylabel("fraud caught")
    plt.title("ROC curve"); plt.legend(fontsize=8)
    plt.tight_layout(); plt.savefig("chart_roc.png", dpi=140); plt.close()

    plt.figure(figsize=(3, 2))
    bins = np.linspace(0, 1, 70)
    plt.hist(S[y_test == 0], bins, alpha=.75, label="legitimate", color="#2a9d8f")
    plt.hist(S[y_test == 1], bins, alpha=.75, label="fraud", color="#e76f51")
    plt.axvline(approve_line, color="k", ls="--"); plt.axvline(block_line, color="k", ls="--")
    plt.yscale("log"); plt.xlabel("risk score S"); plt.ylabel("transactions")
    plt.title("Risk scores, with the two decision lines"); plt.legend()
    plt.tight_layout(); plt.savefig("chart_risk_scores.png", dpi=140); plt.close()

    cm = confusion_matrix(y_test, (S >= best_cut).astype(int))
    plt.figure(figsize=(2.5, 2))
    plt.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            plt.text(j, i, f"{cm[i, j]:,}", ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    plt.xticks([0, 1], ["said legit", "said fraud"])
    plt.yticks([0, 1], ["was legit", "was fraud"])
    plt.title("HELF confusion matrix")
    plt.tight_layout(); plt.savefig("chart_confusion.png", dpi=140); plt.close()

    table.to_csv("results_table.csv", index=False)
    print("          chart_roc.png, chart_risk_scores.png, chart_confusion.png,"
          " results_table.csv")
    print("\nDone.")


if __name__ == "__main__":
    main()


In [ ]:
from IPython.display import Image

print('ROC Curve:')
display(Image('chart_roc.png'))


In [ ]:
print('Risk Scores Distribution:')
display(Image('chart_risk_scores.png'))


In [ ]:
print('Confusion Matrix:')
display(Image('chart_confusion.png'))
